In [1]:
# =============================================================================
# AutoRec FIXED — Replication of Sedhain et al., WWW 2015
# Target: RMSE ~0.831 on MovieLens 1M
#
# Key fixes over v1:
#   FIX 1 — Loss: sum() over observed entries (not mean), matching Eq. 2
#   FIX 2 — Optimizer: FULL-BATCH RProp (paper's exact protocol)
#   FIX 3 — Epochs: 300 (RProp needs more iterations than Adam/SGD)
#   FIX 4 — Default rating = 3 for cold-start test items (paper protocol)
#   FIX 5 — L2 reg on W, V only — biases μ and b are NOT regularised
#   FIX 6 — Raw ratings 1–5 (no normalisation)
#   FIX 7 — DataLoader num_workers=0 (Kaggle compatibility)
# =============================================================================

import os, math, random, zipfile, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =============================================================================
# 1. DATA
# =============================================================================

def download_ml1m(data_dir="./data"):
    os.makedirs(data_dir, exist_ok=True)
    ratings_path = os.path.join(data_dir, "ratings.dat")
    if os.path.exists(ratings_path):
        print("Dataset already present.")
        return ratings_path
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = os.path.join(data_dir, "ml-1m.zip")
    print("Downloading MovieLens 1M …")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)
    src = os.path.join(data_dir, "ml-1m", "ratings.dat")
    os.rename(src, ratings_path)
    os.remove(zip_path)
    print("Done.")
    return ratings_path


def load_ratings(ratings_path):
    data = []
    with open(ratings_path, "r") as f:
        for line in f:
            u, i, r, _ = line.strip().split("::")
            data.append((int(u) - 1, int(i) - 1, float(r)))
    data = np.array(data, dtype=np.float32)
    n_users = int(data[:, 0].max()) + 1
    n_items = int(data[:, 1].max()) + 1
    print(f"Ratings: {len(data):,}  Users: {n_users}  Items: {n_items}")
    return data, n_users, n_items


def train_test_split(ratings, test_ratio=0.10, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(ratings))
    n_test = int(len(ratings) * test_ratio)
    return ratings[idx[n_test:]], ratings[idx[:n_test]]


def build_item_matrix(ratings, n_users, n_items, default_unobserved=0.0):
    """
    R : (n_items, n_users)
    Unobserved entries are 0 (mask will ignore them in loss).
    """
    R = np.full((n_items, n_users), default_unobserved, dtype=np.float32)
    mask = np.zeros((n_items, n_users), dtype=bool)
    for u, i, r in ratings:
        R[int(i), int(u)] = r
        mask[int(i), int(u)] = True
    return R, mask

# =============================================================================
# 2. DATASET  (item-based)
# =============================================================================

class ItemDataset(Dataset):
    def __init__(self, R, mask):
        self.R    = torch.tensor(R,    dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.bool)

    def __len__(self):
        return self.R.shape[0]

    def __getitem__(self, idx):
        return self.R[idx], self.mask[idx]

# =============================================================================
# 3. MODEL
# =============================================================================

class IAutoRec(nn.Module):
    """
    h(r; θ) = f(W · g(V·r + μ) + b)
      g = Sigmoid  (hidden, best per Table 1b)
      f = Identity (output, best per Table 1b)
    """
    def __init__(self, n_users: int, k: int = 500):
        super().__init__()
        self.encoder = nn.Linear(n_users, k)   # V, μ
        self.decoder = nn.Linear(k, n_users)   # W, b
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, r):
        z    = torch.sigmoid(self.encoder(r))
        rhat = self.decoder(z)
        return rhat


def observed_mse_loss(rhat, r, mask):
    """
    FIX 1: Paper Eq. 2 sums squared errors over observed entries.
    Using .sum() (not .mean()) is critical for correct gradient scaling
    when used with full-batch RProp.
    """
    diff = (rhat - r).pow(2)
    return diff[mask].sum()


def l2_reg(model: IAutoRec, lam: float):
    """
    FIX 5: Regularise W (decoder.weight) and V (encoder.weight) ONLY.
    Biases μ and b are NOT regularised (paper Eq. 2).
    """
    return (lam / 2.0) * (
        model.encoder.weight.pow(2).sum() +
        model.decoder.weight.pow(2).sum()
    )

# =============================================================================
# 4. EVALUATION
# =============================================================================

def compute_rmse(model, test_ratings, R_train, n_users, n_items, device):
    """
    FIX 4: Items not seen in training → use default rating vector of all 3s
    (paper: "We use a default rating of 3 for test users or items without
     training observations").
    """
    model.eval()

    # Build full training matrix (all items, even cold ones) on CPU
    default_vec = torch.full((n_users,), 3.0, dtype=torch.float32)

    # Group test entries by item for efficient batch inference
    item_to_tests: dict[int, list] = {}
    for u, i, r in test_ratings:
        i_int = int(i)
        item_to_tests.setdefault(i_int, []).append((int(u), float(r)))

    sq_err, count = 0.0, 0

    with torch.no_grad():
        for i_int, pairs in item_to_tests.items():
            # FIX 4: Use training vector; if item is cold → all-3 vector
            if R_train[i_int].sum() > 0:
                r_vec = torch.tensor(R_train[i_int], dtype=torch.float32)
            else:
                r_vec = default_vec.clone()

            r_vec = r_vec.unsqueeze(0).to(device)   # (1, n_users)
            rhat  = model(r_vec).squeeze(0)          # (n_users,)

            for u_int, r_true in pairs:
                r_pred = float(rhat[u_int].item())
                r_pred = max(1.0, min(5.0, r_pred))  # clip to valid range
                sq_err += (r_pred - r_true) ** 2
                count  += 1

    return math.sqrt(sq_err / count)

# =============================================================================
# 5. TRAINING  (full-batch RProp — exact paper protocol)
# =============================================================================

def train_autorec(
    R_train,          # np.ndarray (n_items, n_users)
    mask_train,       # np.ndarray (n_items, n_users) bool
    test_ratings,     # np.ndarray (N_test, 3)
    n_users, n_items,
    k          = 500,
    lam        = 0.001,
    n_epochs   = 300,
    ckpt_dir   = "./checkpoints",
    run_id     = 0,
    device     = DEVICE,
    patience   = 30,
    ckpt_every = 20,
):
    os.makedirs(ckpt_dir, exist_ok=True)
    best_path = os.path.join(ckpt_dir, f"best_run{run_id}.pt")
    last_path = os.path.join(ckpt_dir, f"last_run{run_id}.pt")

    model = IAutoRec(n_users=n_users, k=k).to(device)

    # ── FIX 2: Full-batch RProp ───────────────────────────────────────────────
    # The paper states RProp is used over the full training set.
    # We load ALL item vectors at once (R_train is ~3960 × 6041 ≈ 24M floats,
    # about 95 MB — comfortably fits in GPU VRAM).
    # If you hit OOM, fall back to the mini-batch version below.
    R_tensor    = torch.tensor(R_train,    dtype=torch.float32, device=device)
    mask_tensor = torch.tensor(mask_train, dtype=torch.bool,    device=device)

    # FIX 3: Rprop parameters tuned for full-batch convergence
    optimizer = optim.Rprop(
        model.parameters(),
        lr       = 0.01,     # larger initial step for full-batch
        etas     = (0.5, 1.2),
        step_sizes = (1e-6, 50),
    )

    best_rmse  = float("inf")
    no_improve = 0
    history    = {"loss": [], "rmse": []}

    print(f"\n{'='*62}")
    print(f"Run {run_id+1} | k={k} | λ={lam} | epochs={n_epochs} | {device}")
    print(f"{'='*62}")

    bar = tqdm(range(1, n_epochs + 1), desc="Training", unit="ep")
    for epoch in bar:
        model.train()
        optimizer.zero_grad()

        # Full-batch forward pass
        rhat = model(R_tensor)                          # (n_items, n_users)
        loss = observed_mse_loss(rhat, R_tensor, mask_tensor) + \
               l2_reg(model, lam)
        loss.backward()
        optimizer.step()

        history["loss"].append(loss.item())

        # ── Validate every epoch ─────────────────────────────────────────────
        val_rmse = compute_rmse(
            model, test_ratings, R_train, n_users, n_items, device)
        history["rmse"].append(val_rmse)

        bar.set_postfix({"loss": f"{loss.item():.1f}",
                         "RMSE": f"{val_rmse:.4f}"})

        # ── Checkpoint every N epochs ─────────────────────────────────────────
        if epoch % ckpt_every == 0:
            torch.save({
                "epoch": epoch, "val_rmse": val_rmse,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "config": {"k": k, "lam": lam, "n_users": n_users, "n_items": n_items},
            }, last_path)
            tqdm.write(f"  [ckpt] epoch {epoch} | RMSE {val_rmse:.4f} → {last_path}")

        # ── Best model ────────────────────────────────────────────────────────
        if val_rmse < best_rmse:
            best_rmse  = val_rmse
            no_improve = 0
            torch.save({
                "epoch": epoch, "val_rmse": val_rmse,
                "model_state": model.state_dict(),
                "config": {"k": k, "lam": lam, "n_users": n_users, "n_items": n_items},
            }, best_path)
        else:
            no_improve += 1

        if no_improve >= patience:
            tqdm.write(f"\n  Early stop at epoch {epoch} (patience={patience})")
            break

    print(f"\n  Best RMSE (run {run_id+1}): {best_rmse:.4f}")
    print(f"  Saved → {best_path}")
    return best_rmse, history


# ── Mini-batch fallback (use if GPU OOM on full-batch) ────────────────────────
def train_autorec_minibatch(
    R_train, mask_train, test_ratings, n_users, n_items,
    k=500, lam=0.001, n_epochs=500, batch_size=512,
    ckpt_dir="./checkpoints", run_id=0, device=DEVICE, patience=30,
):
    """
    Mini-batch version. Slightly less faithful to the paper but handles
    large datasets. Uses Adam instead of RProp (RProp is ill-defined
    for mini-batch because step magnitudes vary per batch).
    """
    os.makedirs(ckpt_dir, exist_ok=True)
    best_path = os.path.join(ckpt_dir, f"best_mb_run{run_id}.pt")
    dataset   = ItemDataset(R_train, mask_train)
    loader    = DataLoader(dataset, batch_size=batch_size,
                           shuffle=True, num_workers=0)

    model     = IAutoRec(n_users=n_users, k=k).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=0)

    best_rmse, no_improve = float("inf"), 0
    bar = tqdm(range(1, n_epochs + 1), desc="Mini-batch", unit="ep")
    for epoch in bar:
        model.train()
        total = 0.0
        for r_b, m_b in loader:
            r_b, m_b = r_b.to(device), m_b.to(device)
            optimizer.zero_grad()
            rhat = model(r_b)
            loss = observed_mse_loss(rhat, r_b, m_b) + l2_reg(model, lam)
            loss.backward()
            optimizer.step()
            total += loss.item()

        val_rmse = compute_rmse(
            model, test_ratings, R_train, n_users, n_items, device)
        bar.set_postfix({"RMSE": f"{val_rmse:.4f}"})

        if val_rmse < best_rmse:
            best_rmse, no_improve = val_rmse, 0
            torch.save({"epoch": epoch, "val_rmse": val_rmse,
                        "model_state": model.state_dict()}, best_path)
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    print(f"\n  Best RMSE: {best_rmse:.4f}")
    return best_rmse, {}

# =============================================================================
# 6. MAIN
# =============================================================================

def main():
    ratings_path = download_ml1m()
    ratings, n_users, n_items = load_ratings(ratings_path)

    # Paper hyperparameters for ML-1M
    K       = 500
    LAM     = 0.001
    EPOCHS  = 300
    N_RUNS  = 5

    all_rmse = []

    for run in range(N_RUNS):
        seed = SEED + run * 17
        train_r, test_r = train_test_split(ratings, test_ratio=0.10, seed=seed)
        R_train, mask_train = build_item_matrix(train_r, n_users, n_items)

        best_rmse, _ = train_autorec(
            R_train     = R_train,
            mask_train  = mask_train,
            test_ratings = test_r,
            n_users     = n_users,
            n_items     = n_items,
            k           = K,
            lam         = LAM,
            n_epochs    = EPOCHS,
            ckpt_dir    = "./checkpoints",
            run_id      = run,
            device      = DEVICE,
            patience    = 30,
            ckpt_every  = 20,
        )
        all_rmse.append(best_rmse)

    mean_r = np.mean(all_rmse)
    std_r  = np.std(all_rmse)
    ci_95  = 1.96 * std_r / math.sqrt(N_RUNS)

    print("\n" + "="*62)
    print("FINAL RESULTS (5-run average, paper protocol)")
    print("="*62)
    for i, r in enumerate(all_rmse):
        print(f"  Run {i+1}: RMSE = {r:.4f}")
    print(f"\n  Mean RMSE : {mean_r:.4f}")
    print(f"  Std       : {std_r:.4f}")
    print(f"  95% CI    : ±{ci_95:.4f}")
    print(f"\n  Paper target (I-AutoRec, ML-1M): 0.831")
    print("="*62)
    return mean_r


if __name__ == "__main__":
    main()

Using device: cuda
Done.
Ratings: 1,000,209  Users: 6040  Items: 3952

Run 1 | k=500 | λ=0.001 | epochs=300 | cuda


Training:   7%|▋         | 20/300 [00:54<12:45,  2.73s/ep, loss=620584.1, RMSE=0.8854] 

  [ckpt] epoch 20 | RMSE 0.8854 → ./checkpoints/last_run0.pt


Training:  13%|█▎        | 40/300 [01:47<11:40,  2.69s/ep, loss=301542.4, RMSE=0.9330]

  [ckpt] epoch 40 | RMSE 0.9330 → ./checkpoints/last_run0.pt


...

  [ckpt] epoch 20 | RMSE 0.8888 → ./checkpoints/last_run4.pt


Training:  13%|█▎        | 40/300 [01:46<11:28,  2.65s/ep, loss=319224.3, RMSE=0.9346]

  [ckpt] epoch 40 | RMSE 0.9346 → ./checkpoints/last_run4.pt


Training:  18%|█▊        | 54/300 [02:25<11:02,  2.69s/ep, loss=211054.4, RMSE=0.9955]


  Early stop at epoch 55 (patience=30)

  Best RMSE (run 5): 0.8786
  Saved → ./checkpoints/best_run4.pt

FINAL RESULTS (5-run average, paper protocol)
  Run 1: RMSE = 0.8759
  Run 2: RMSE = 0.8824
  Run 3: RMSE = 0.8763
  Run 4: RMSE = 0.8761
  Run 5: RMSE = 0.8786

  Mean RMSE : 0.8779
  Std       : 0.0025
  95% CI    : ±0.0022

  Paper target (I-AutoRec, ML-1M): 0.831
